# Milestone 5

Ensemble inference, Test-Time Augmentation (TTA), and confidence analysis using fine-tuned DeBERTa and RoBERTa sequence classification models.

## Setup run

In [1]:
import os, numpy as np, pandas as pd, torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
LABEL_MAP   = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

In [2]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device: {DEVICE}")

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv
Device: cuda


## Load data

In [3]:
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")

print(f"train: {train.shape}  test: {test.shape}")

train: (2000, 8)  test: (500, 7)


## Load fine-tuned models

Both models are loaded as `AutoModelForSequenceClassification` with `num_labels=5`,
matching the label mapping: 0=A, 1=B, 2=C, 3=D, 4=E.

In [4]:
# DeBERTa
DEBERTA_MODEL = "microsoft/deberta-v3-small"

deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL)
deberta_model     = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_MODEL, num_labels=5
).to(DEVICE)
deberta_model.eval()
print("DeBERTa loaded.")

# RoBERTa
ROBERTA_MODEL = "roberta-base"

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL)
roberta_model     = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_MODEL, num_labels=5
).to(DEVICE)
roberta_model.eval()
print("RoBERTa loaded.")

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa loaded.


## Inference helpers

Two utility functions used throughout the notebook: one that runs a single model on a prompt string and returns 5 softmax probabilities, and another that builds the formatted input string from a row.

In [5]:
def get_probs(prompt_str, tokenizer, model, max_length=256):
    """Tokenize prompt_str, run model, return softmax probs as numpy array (5,)."""
    enc = tokenizer(
        prompt_str,
        return_tensors  = "pt",
        truncation      = True,
        max_length      = max_length,
        padding         = True,
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits[0]   # shape (5,)
    return F.softmax(logits, dim=-1).cpu().numpy()


def build_prompt(row):
    """Concatenate prompt and all 5 options into a single classification input."""
    opts = " ".join([f"{opt}: {str(row[opt])}" for opt in OPTION_COLS])
    return f"{str(row['prompt'])} {opts}"


def top3_string(probs):
    """Return top-3 option letters ranked by probability, space-separated."""
    ranked = np.argsort(-probs)[:3]
    return " ".join([LABEL_MAP[i] for i in ranked])


def top1_letter(probs):
    return LABEL_MAP[int(np.argmax(probs))]

## Q1. DeBERTa top-1 option and probability for row index 25

In [6]:
row25        = train.iloc[25]
prompt25     = build_prompt(row25)

deberta_probs25 = get_probs(prompt25, deberta_tokenizer, deberta_model)

print("DeBERTa probabilities for row 25:")
for i, prob in enumerate(deberta_probs25):
    print(f"  {LABEL_MAP[i]}: {prob:.6f}")

q1_letter = top1_letter(deberta_probs25)
q1_prob   = round(float(deberta_probs25[np.argmax(deberta_probs25)]), 4)
print(f"\nQ1 answer — Top-1: {q1_letter}, probability: {q1_prob}")
print(f"True answer: {row25['answer']}")

DeBERTa probabilities for row 25:
  A: 0.239014
  B: 0.123352
  C: 0.268311
  D: 0.198486
  E: 0.171021

Q1 answer — Top-1: C, probability: 0.2683
True answer: E


## Q2. Simple average ensemble — top-1 option for row 25

In [7]:
roberta_probs25 = get_probs(prompt25, roberta_tokenizer, roberta_model)

print("RoBERTa probabilities for row 25:")
for i, prob in enumerate(roberta_probs25):
    print(f"  {LABEL_MAP[i]}: {prob:.6f}")

avg_probs25 = (deberta_probs25 + roberta_probs25) / 2

print("\nAveraged probabilities:")
for i, prob in enumerate(avg_probs25):
    print(f"  {LABEL_MAP[i]}: {prob:.6f}")

q2_letter = top1_letter(avg_probs25)
print(f"\nQ2 answer — Top-1 after simple average ensemble: {q2_letter}")

RoBERTa probabilities for row 25:
  A: 0.198269
  B: 0.214327
  C: 0.186432
  D: 0.205179
  E: 0.195794

Averaged probabilities:
  A: 0.218641
  B: 0.168839
  C: 0.227371
  D: 0.201833
  E: 0.183407

Q2 answer — Top-1 after simple average ensemble: C


## Q3. Weighted ensemble (DeBERTa 0.70 / RoBERTa 0.30) — top-1 option for row 25

In [8]:
W_DEBERTA = 0.70
W_ROBERTA = 0.30

weighted_probs25 = W_DEBERTA * deberta_probs25 + W_ROBERTA * roberta_probs25

print("Weighted ensemble probabilities (0.70 DeBERTa / 0.30 RoBERTa):")
for i, prob in enumerate(weighted_probs25):
    print(f"  {LABEL_MAP[i]}: {prob:.6f}")

q3_letter = top1_letter(weighted_probs25)
print(f"\nQ3 answer — Top-1 after weighted ensemble: {q3_letter}")

Weighted ensemble probabilities (0.70 DeBERTa / 0.30 RoBERTa):
  A: 0.226839
  B: 0.150663
  C: 0.243796
  D: 0.200592
  E: 0.178489

Q3 answer — Top-1 after weighted ensemble: C


## Q4. Top-3 prediction string for row 25 (weighted ensemble)

In [9]:
q4_answer = top3_string(weighted_probs25)
print(f"Q4 answer — Top-3 prediction string: {q4_answer}")
print(f"True answer: {row25['answer']}") 

Q4 answer — Top-3 prediction string: C A D
True answer: E


## Q5. Run weighted ensemble on full test.csv — count prediction rows in submission.csv

In [10]:
print(f"Running weighted ensemble on {len(test)} test rows...")

predictions = []
for i, (_, row) in enumerate(test.iterrows()):
    prompt_i        = build_prompt(row)
    deb_probs       = get_probs(prompt_i, deberta_tokenizer, deberta_model)
    rob_probs       = get_probs(prompt_i, roberta_tokenizer, roberta_model)
    weighted_probs  = W_DEBERTA * deb_probs + W_ROBERTA * rob_probs
    predictions.append(top3_string(weighted_probs))

    if i % 100 == 0:
        print(f"  {i}/{len(test)} done...")

submission = pd.DataFrame({
    'ID'         : test['id'],
    'Prediction' : predictions
})

submission.to_csv('/kaggle/working/submission.csv', index=False)

q5_answer = len(submission)
print(f"\nsubmission.csv saved.")
print(f"Q5 answer — prediction rows (excluding header): {q5_answer}")
submission.head()

Running weighted ensemble on 500 test rows...
  0/500 done...
  100/500 done...
  200/500 done...
  300/500 done...
  400/500 done...

submission.csv saved.
Q5 answer — prediction rows (excluding header): 500


,ID,Prediction
0,1,A C D
1,2,A D E
2,3,C A D
3,4,A D C
4,5,A B C


## Q6. Test-Time Augmentation (TTA) -> how many of first 50 test rows get a different Top-1

Two passes per row with DeBERTa:
1. Original prompt
2. Prepend `"Answer the following multiple-choice question carefully:"`

Average the two probability arrays, check if Top-1 changes.

In [11]:
TTA_PREFIX = "Answer the following multiple-choice question carefully: "

tta_changes = 0

for i in range(50):
    row = test.iloc[i]

    # Pass 1 — original prompt
    prompt_orig = build_prompt(row)
    probs_orig  = get_probs(prompt_orig, deberta_tokenizer, deberta_model)

    # Pass 2 — instruction-augmented prompt
    prompt_aug  = TTA_PREFIX + build_prompt(row)
    probs_aug   = get_probs(prompt_aug, deberta_tokenizer, deberta_model)

    # Average both passes
    probs_tta   = (probs_orig + probs_aug) / 2

    top1_before = top1_letter(probs_orig)
    top1_after  = top1_letter(probs_tta)

    if top1_before != top1_after:
        tta_changes += 1
        print(f"  Row {i:2d}: {top1_before} → {top1_after}  (changed)")

q6_answer = tta_changes
print(f"\nQ6 answer — rows with different Top-1 after TTA: {q6_answer} / 50")

  Row  2: C → A  (changed)
  Row 12: C → A  (changed)
  Row 16: C → A  (changed)
  Row 17: E → A  (changed)
  Row 18: C → A  (changed)
  Row 23: C → A  (changed)
  Row 25: C → A  (changed)
  Row 26: C → A  (changed)
  Row 30: C → A  (changed)
  Row 32: C → A  (changed)
  Row 33: E → A  (changed)
  Row 48: C → A  (changed)

Q6 answer — rows with different Top-1 after TTA: 12 / 50


## Q7. DeBERTa vs Weighted Ensemble — rows with different Top-1 in first 100 test rows

In [12]:
diff_top1 = 0
q7_details = []

for i in range(100):
    row = test.iloc[i]
    prompt_i = build_prompt(row)

    deb_probs      = get_probs(prompt_i, deberta_tokenizer, deberta_model)
    rob_probs      = get_probs(prompt_i, roberta_tokenizer, roberta_model)
    weighted_probs = W_DEBERTA * deb_probs + W_ROBERTA * rob_probs

    t1_deb = top1_letter(deb_probs)
    t1_ens = top1_letter(weighted_probs)

    if t1_deb != t1_ens:
        diff_top1 += 1
        q7_details.append((i, t1_deb, t1_ens))

print("Rows where DeBERTa and Ensemble Top-1 differ:")
for row_i, d, e in q7_details:
    print(f"  Row {row_i:3d}: DeBERTa={d}  Ensemble={e}")

q7_answer = diff_top1
print(f"\nQ7 answer — rows with different Top-1: {q7_answer} / 100")

Rows where DeBERTa and Ensemble Top-1 differ:
  Row  12: DeBERTa=C  Ensemble=E
  Row  50: DeBERTa=A  Ensemble=D
  Row  70: DeBERTa=C  Ensemble=A

Q7 answer — rows with different Top-1: 3 / 100


## Q8. Confidence Gain — rows with positive Ensemble − DeBERTa confidence in first 100 test rows

In [13]:
positive_gain = 0

for i in range(100):
    row = test.iloc[i]
    prompt_i = build_prompt(row)

    deb_probs      = get_probs(prompt_i, deberta_tokenizer, deberta_model)
    rob_probs      = get_probs(prompt_i, roberta_tokenizer, roberta_model)
    weighted_probs = W_DEBERTA * deb_probs + W_ROBERTA * rob_probs

    conf_deb = float(np.max(deb_probs))
    conf_ens = float(np.max(weighted_probs))
    gain     = conf_ens - conf_deb

    if gain > 0:
        positive_gain += 1

q8_answer = positive_gain
print(f"Q8 answer — rows with positive confidence gain: {q8_answer} / 100")

Q8 answer — rows with positive confidence gain: 0 / 100


## Q9. Top-3 string changes — rows where ordered Top-3 differs between DeBERTa and Ensemble (first 100 test rows)

In [14]:
top3_changes = 0

for i in range(100):
    row = test.iloc[i]
    prompt_i = build_prompt(row)

    deb_probs      = get_probs(prompt_i, deberta_tokenizer, deberta_model)
    rob_probs      = get_probs(prompt_i, roberta_tokenizer, roberta_model)
    weighted_probs = W_DEBERTA * deb_probs + W_ROBERTA * rob_probs

    t3_deb = top3_string(deb_probs)
    t3_ens = top3_string(weighted_probs)

    if t3_deb != t3_ens:
        top3_changes += 1

q9_answer = top3_changes
print(f"Q9 answer — rows with at least one change in ordered Top-3: {q9_answer} / 100")

Q9 answer — rows with at least one change in ordered Top-3: 23 / 100


## Q10. MAP@3 on first 100 validation samples using weighted ensemble

Using the first 100 rows of `train.csv` (which have known answers) as the validation set.

In [15]:
def map_at_3(true_answer, top3_str):
    top3 = top3_str.split()
    if true_answer in top3:
        return 1 / (top3.index(true_answer) + 1)
    return 0

map3_scores = []

print("Running weighted ensemble on first 100 train rows for MAP@3...")
for i in range(100):
    row = train.iloc[i]
    prompt_i = build_prompt(row)

    deb_probs      = get_probs(prompt_i, deberta_tokenizer, deberta_model)
    rob_probs      = get_probs(prompt_i, roberta_tokenizer, roberta_model)
    weighted_probs = W_DEBERTA * deb_probs + W_ROBERTA * rob_probs

    t3_ens = top3_string(weighted_probs)
    score  = map_at_3(str(row['answer']), t3_ens)
    map3_scores.append(score)

q10_answer = round(float(np.mean(map3_scores)), 4)
print(f"\nQ10 answer — MAP@3 on first 100 validation rows: {q10_answer}")

Running weighted ensemble on first 100 train rows for MAP@3...

Q10 answer — MAP@3 on first 100 validation rows: 0.3667
